# 🐭 go or nogo? — the real neuron, on your own

**Time:** ~45 minutes for the analysis, then stretch tasks for as long as you like.

## The experiment

This data comes from **Miguel Maravall's lab here at Sussex** ([Bale et al., 2021, *Current Biology*](https://doi.org/10.1016/j.cub.2020.10.059)). A mouse feels one of two whisker "words" — **go** (lick, and get water) or **nogo** (don't lick) — while a microscope records one neuron's activity.

**The question:** does this neuron tell go from nogo? **Show it.**

## How this notebook works

You did every step on small data in notebook 1. Here there are **no skeletons** — only **four checkpoints**. Each one says which variables to make; how you get there is up to you. The self-check under each tells you ✅ when you're on track.

Stuck? Each checkpoint has a **💡 Hint** you can open. Try without it first.

What you need to know about the real data:

- every word lasts exactly **9 frames**
- use a window of **10 frames before** each start and **30 from it on** — 40 frames
- the go/nogo columns are `1.0` while the word plays, `0.0` otherwise

Then **Part 2** — stretch tasks — removes the shortcuts this analysis takes, and ends with a statistical test.

## Our data

Run this first: `dff`, `go` and `nogo` for **neuron 1**, the neuron from the lecture. Neuron 2 is downloaded for Checkpoint 4.

In [ ]:
# Run this first. It loads two neurons' recordings from the lecture —
# from data/ if you downloaded them, otherwise straight from GitHub.
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab05/data/"
DATA = Path("data")            # same layout as a uv-start project
DATA.mkdir(exist_ok=True)
NEURON_1 = DATA / "neuron_0588_20180828_5_lecture.csv"
NEURON_2 = DATA / "neuron_0406_20180620_8_lecture.csv"
for path in [NEURON_1, NEURON_2]:
    if not path.exists():
        urllib.request.urlretrieve(DATA_URL + path.name, path)

# Columns: 0 frame, 1 dff, 2 go, 3 nogo, 4 reward, 5 lick
# unpack=True hands back each column as its own array - three loads in one line
dff, go, nogo = np.loadtxt(NEURON_1, delimiter=",", skiprows=1, usecols=(1, 2, 3), unpack=True)

FRAME_RATE = 10.8   # frames per second
print("neuron 1:", dff.size, "frames")

## Checkpoint 1 — where does every trial start?

Make `go_starts` and `nogo_starts`: arrays holding the **first frame** of every go and every nogo trial.

<details>
<summary>💡 Hint</summary>

Notebook 1, steps 3 and 4: `np.where(...)[0]` gives every frame where the word is on, in blocks of 9. You want the first of each block.
</details>

In [ ]:
# Your code — make go_starts and nogo_starts


# --- self-check (do not edit) ---
assert "go_starts" in globals() and "nogo_starts" in globals(), "❌ Make go_starts and nogo_starts"
assert len(go_starts) != 2286, "❌ go_starts holds every frame where go is on — you want only the first of each block of 9"
assert len(go_starts) == 254, f"❌ There are 254 go trials, you have {len(go_starts)}"
assert len(nogo_starts) == 264, f"❌ There are 264 nogo trials, you have {len(nogo_starts)}"
assert (go[go_starts] == 1).all() and (go[np.asarray(go_starts) - 1] == 0).all(), "❌ Some of go_starts aren't the FIRST frame of a word"
assert (nogo[nogo_starts] == 1).all() and (nogo[np.asarray(nogo_starts) - 1] == 0).all(), "❌ Some of nogo_starts aren't the FIRST frame of a word"
print("✅ Checkpoint 1 — 254 go and 264 nogo trials")

## Checkpoint 2 — every trial, in a grid

Make `go_trials` and `nogo_trials`: 2-D arrays with one **row per trial** and one **column per moment** — the 40-frame window round each start.

Look closely at the first few starts before you begin.

<details>
<summary>💡 Hint</summary>

Notebook 1, steps 5 and 6: `dff[start - 10 : start + 30]` for every start, collected in a list, then `np.array()`.

The first nogo trial starts at frame 2 — there aren't 10 frames before it. What does `dff[-8:32]` give? Skip that trial.
</details>

In [ ]:
# Your code — make go_trials and nogo_trials


# --- self-check (do not edit) ---
assert "go_trials" in globals() and "nogo_trials" in globals(), "❌ Make go_trials and nogo_trials"
assert np.ndim(go_trials) == 2, "❌ go_trials should be 2-D — a list of windows, turned into an array with np.array()"
assert np.shape(go_trials) == (254, 40), f"❌ go_trials should be (254, 40), got {np.shape(go_trials)}"
assert np.shape(nogo_trials) != (264, 40), "❌ nogo_trials has 264 rows — but the first trial has no 10 frames before it. Look at nogo_trials[0]"
assert np.shape(nogo_trials) == (263, 40), f"❌ nogo_trials should be (263, 40), got {np.shape(nogo_trials)}"
assert np.allclose(go_trials[0], dff[33:73]), "❌ go_trials[0] should be frames 33-72: 10 before the first start (43), 30 from it"
print("✅ Checkpoint 2 — two grids, (254, 40) and (263, 40)")

## Checkpoint 3 — the average trial

Make `go_response` and `nogo_response` — the **average** go and nogo trial, 40 numbers each — and their peaks **from the start of the word onwards**, rounded to 3 decimal places, in `go_peak` and `nogo_peak`.

<details>
<summary>💡 Hint</summary>

Notebook 1, step 7: `axis=0` averages down the columns. The word starts at column 10, so the peak is the max of `[10:]`.
</details>

In [ ]:
# Your code — go_response, nogo_response, go_peak, nogo_peak


# --- self-check (do not edit) ---
for name in ["go_response", "nogo_response", "go_peak", "nogo_peak"]:
    assert name in globals(), f"❌ Make {name}"
assert np.shape(go_response) != (254,), "❌ go_response has one value per TRIAL — that's axis=1. You want one per moment"
assert np.shape(go_response) == (40,) and np.shape(nogo_response) == (40,), "❌ Each response should have 40 values, one per moment"
assert go_peak == 0.29, f"❌ go_peak should be 0.29, got {go_peak} — the max from column 10 on, rounded to 3"
assert nogo_peak == 0.057, f"❌ nogo_peak should be 0.057, got {nogo_peak}"
print("✅ Checkpoint 3 — go peaks", round(go_peak / nogo_peak, 1), "times higher than nogo")

Run this to see your answer. (Plotting comes in Lab 7.)

In [ ]:
# Run me - one trial, and your two averages
seconds = np.arange(-10, 30) / FRAME_RATE

plt.figure(figsize=(8, 3.5))
plt.plot(seconds, go_trials[0], color="lightgrey", label="one go trial")
plt.plot(seconds, go_response, lw=2.5, label=f"go — {len(go_trials)} trials averaged")
plt.plot(seconds, nogo_response, lw=2.5, label=f"nogo — {len(nogo_trials)} trials averaged")
plt.axvline(0, color="black", lw=1)
plt.axvspan(0, 9 / FRAME_RATE, alpha=0.1)
plt.xlabel("seconds from the start of the word")
plt.ylabel("activity (dF/F)")
plt.legend()
plt.show()

After **go**, activity climbs as the word ends and stays up; after **nogo** it stays flat. Neuron 1 tells the words apart.

> 🧠 On go trials the mouse also **licks** and gets **water**. Is the neuron answering the *word* or the *licking*? Worth thinking about — the file has a `lick` column (5).

## Checkpoint 4 — any neuron, in one line

1. Write `response_to(trace, marker)`: give it a neuron's activity and a go or nogo column, get back the average trial (40 numbers).
2. Load neuron 2 into `dff_2`, `go_2` and `nogo_2`.
3. Store each neuron's go-peak ÷ nogo-peak, rounded to 1 decimal place, in `ratio_1` and `ratio_2`.

<details>
<summary>💡 Hint</summary>

The function is Checkpoints 1–3 joined up — but use `trace` and `marker` inside it, never `dff` and `go`, or it will only ever analyse neuron 1. The data cell shows how to load three columns in one line.
</details>

In [ ]:
# Your code — response_to, neuron 2, ratio_1 and ratio_2


# --- self-check (do not edit) ---
assert "response_to" in globals(), "❌ No function called response_to yet"
assert response_to(dff, go) is not None, "❌ response_to gives back None — did you forget to return?"
assert np.shape(response_to(dff, go)) == (40,), f"❌ response_to should return 40 values, got shape {np.shape(response_to(dff, go))}"
assert np.allclose(response_to(dff, go), go_response), "❌ response_to(dff, go) doesn't match go_response"
assert not np.allclose(response_to(dff, nogo), go_response), "❌ response_to(dff, nogo) gives the GO answer — inside the function use trace and marker, not dff and go"
assert np.allclose(response_to(dff, nogo), nogo_response), "❌ response_to(dff, nogo) doesn't match nogo_response — is the trial at frame 2 skipped?"
assert np.shape(dff_2) == (9635,) and np.shape(go_2) == (9635,) and np.shape(nogo_2) == (9635,), "❌ Load neuron 2's columns 1, 2 and 3 into dff_2, go_2 and nogo_2"
assert ratio_1 == 5.1, f"❌ ratio_1 should be 5.1, got {ratio_1}"
assert ratio_2 == 1.2, f"❌ ratio_2 should be 1.2, got {ratio_2}"
print("✅ Checkpoint 4 — neuron 1 prefers go 5.1 to 1; neuron 2 barely cares (1.2)")

## The verdict

**Neuron 1** cares which word it heard. **Neuron 2** fires hard after *both* — it responds to the whisker touch, whatever the word. Same code, two personalities.

---

# Part 2 — Stretch tasks: remove the shortcuts

The analysis above rests on three shortcuts a reviewer would question:

- `[::9]` **assumed** every word lasts exactly 9 frames. Does it?
- The `for` loop cut the windows one at a time. numpy can cut all of them at once.
- "5 times bigger" is not a test. Could **chance** give a difference that big?

These stretch tasks remove all three. Along the way: `np.diff`, indexing with arrays, **broadcasting**, and a statistics test built from shuffles.

They reuse your variables, so run the checkpoints first. The cell below keeps the loop version from Part 1, to check the new way against.

In [ ]:
# Run me - Part 1's way of doing it, kept as reference functions
def starts_by_nine(marker):
    """The first frame of each trial, assuming every word is exactly 9 frames."""
    return np.where(marker == 1)[0][::9]

def trials_by_loop(trace, starts):
    """A (trials x 40) grid: 10 frames before each start, 30 after."""
    trials = []
    for i in starts:
        if i >= 10:
            trials.append(trace[i - 10 : i + 30])
    return np.array(trials)

print(trials_by_loop(dff, starts_by_nine(go)).shape)

## Stretch 1 — `np.diff`: find where things start

`np.diff(a)` subtracts each value from the next one: `a[1] - a[0]`, `a[2] - a[1]`, … On a 0/1 marker the answer is `1` where it switches **on**, `-1` where it switches **off**, and `0` everywhere else:

```
marker          0  0  1  1  1  0  0  1  1  0
np.diff         0  1  0  0 -1  0  1  0 -1
                   ↑        ↑     ↑     ↑
                   on       off   on    off
```

`np.diff` is one shorter than the marker, and `np.diff(a)[i]` compares position `i + 1` with position `i`. So where the diff is `1`, the switch-on happened at **`i + 1`**.

In [ ]:
# DEMO - switch-on and switch-off on a short marker
marker = np.array([0, 0, 1, 1, 1, 0, 0, 1, 1, 0])
print(np.diff(marker))
print("on at: ", np.where(np.diff(marker) == 1)[0] + 1)    # +1: the frame AFTER the jump
print("off at:", np.where(np.diff(marker) == -1)[0] + 1)   # first frame that is 0 again

### 🐞 Broken — one frame early

This finds the go starts with `np.diff` — but every one of them is wrong. No error; the self-check catches it.

> 💡 Print `go[go_starts[:5]]`. A real start should be 1.

In [ ]:
# 🐞 BROKEN — every go start, found with np.diff
go_starts = np.where(np.diff(go) == 1)[0]
print(go_starts[:5])


# --- self-check (do not edit) ---
assert (go[go_starts] == 1).all(), "❌ go is 0 at your 'starts' — they're the frame BEFORE each switch-on. np.diff(go)[i] compares frame i+1 with frame i, so add 1"
assert (go[go_starts - 1] == 0).all(), "❌ go is already on just before some of your starts — those aren't switch-ons"
assert np.array_equal(go_starts, starts_by_nine(go)), "❌ go_starts should match Part 1's starts"
print("✅ Fixed —", len(go_starts), "go starts, identical to Part 1's — but with no assumption about how long a word lasts")

### 🔬 Write it — test the assumption

Now check the shortcut from Part 1.

**Your task:**

1. Make `go_ends` — the **last** frame of every go word. The diff is `-1` there: the marker switches off *after* it.
2. Make `durations` — how many frames each go word lasts, from `go_starts` and `go_ends`. It's an array: one duration per trial, no loop.
3. Store whether **every** duration is 9 in `all_nine` — `True` or `False`.

> 💡 For 1: where the diff is `-1` at position `i`, frame `i` is still on and frame `i + 1` is off — so the last frame is just `i`.
> For 3: an array has `.all()` — "is everything True?", from the lecture's menu.

In [ ]:
# TODO — go_ends, durations and all_nine


# --- self-check (do not edit) ---
assert "go_ends" in globals() and "durations" in globals() and "all_nine" in globals(), "❌ Make go_ends, durations and all_nine"
assert len(go_ends) == len(go_starts), f"❌ There should be one end per start ({len(go_starts)}), got {len(go_ends)}"
assert (go[go_ends] == 1).all(), "❌ go is 0 at some of your ends — they're one frame late. The last frame ON is i, not i + 1"
assert (go[go_ends + 1] == 0).all(), "❌ go is still on after some of your ends"
assert np.array_equal(durations, go_ends - go_starts + 1), "❌ durations should be go_ends - go_starts + 1 — a word from frame 43 to 51 lasts 9 frames, not 8"
assert all_nine == True, f"❌ all_nine should be True — check durations, got {all_nine}"
print("✅ Correct — every go word is exactly 9 frames, so [::9] was safe. Now you've checked it instead of assuming it.")

## Stretch 2 — Cut every window at once

An array of **positions** can index an array — you get those values back in one go:

```python
dff[[43, 125, 166]]      # three frames, picked by position
```

A **2-D** array of positions gives back a **2-D** array of values, in the same shape. So if we build a grid of positions — one row per trial, one column per moment — `dff[grid]` *is* the trial table. No loop.

The grid comes from adding a **column** of starts to a **row** of offsets. numpy stretches each one to match the other — that is **broadcasting**, and Stretch 3 is all about it:

```
starts[:, np.newaxis]     np.arange(-2, 3)          grid
    [[ 43],             [-2 -1  0  1  2]     [[ 41  42  43  44  45]
     [125],        +                     =    [123 124 125 126 127]
     [166]]                                   [164 165 166 167 168]]
```

`np.newaxis` turns a row of 3 values into a column: shape `(3,)` becomes `(3, 1)`.

In [ ]:
# DEMO - a small grid first, then all 254 go trials at once
starts = np.array([43, 125, 166])
grid = starts[:, np.newaxis] + np.arange(-2, 3)
print(grid)
print(dff[grid].round(2))           # the values at those positions, same shape

offsets = np.arange(-10, 30)        # 10 before, 30 after - the window from Part 1
go_trials = dff[go_starts[:, np.newaxis] + offsets]
print(go_trials.shape)
print("same as the loop?", np.array_equal(go_trials, trials_by_loop(dff, go_starts)))

### 🔬 Write it — nogo, without a loop

The first nogo trial starts at frame 2, which broke the loop in Part 1. Here it doesn't crash — it is **worse**: position `2 - 10 = -8` silently wraps around to the **end** of the recording, and that trial gets 8 frames of garbage.

**Your task:**

1. Make `nogo_starts` with `np.diff`.
2. Keep only the starts that are **10 or more**, with a mask. Store them in `nogo_ok`.
3. Make `nogo_trials` from `nogo_ok`, without a loop.

In [ ]:
# TODO — nogo_starts, nogo_ok and nogo_trials — no for loop


# --- self-check (do not edit) ---
assert "nogo_starts" in globals() and "nogo_ok" in globals() and "nogo_trials" in globals(), "❌ Make nogo_starts, nogo_ok and nogo_trials"
assert np.array_equal(nogo_starts, starts_by_nine(nogo)), "❌ nogo_starts doesn't match — np.where(np.diff(nogo) == 1)[0] + 1"
assert nogo_ok.min() >= 10, "❌ nogo_ok still has a start below 10 — keep nogo_starts[nogo_starts >= 10]"
assert len(nogo_ok) == 263, f"❌ nogo_ok should keep 263 starts, got {len(nogo_ok)}"
assert nogo_trials.shape == (263, 40), f"❌ nogo_trials should be (263, 40), got {nogo_trials.shape}"
assert np.array_equal(nogo_trials, trials_by_loop(dff, nogo_starts)), "❌ nogo_trials doesn't match the loop version"
print("✅ Correct — 263 nogo trials in one line, identical to the loop")

## Stretch 3 — Broadcasting: each trial minus its own baseline

Each trial starts from a slightly different level — the neuron was a bit more or less active before the word. To compare trials fairly, subtract each trial's **own baseline**: the mean of its 10 frames **before** the start.

```python
baseline = go_trials[:, :10].mean(axis=1)    # one number per trial: shape (254,)
```

Now subtract it. This is where **broadcasting** matters. numpy lines shapes up **from the right**, and two sizes fit if they are equal or one of them is 1:

| grid | minus | works? | what happens |
|---|---|---|---|
| `(3, 5)` | `(5,)` | ✅ | one value per **column**, reused down every row |
| `(3, 5)` | `(3,)` | ❌ | 5 and 3 don't match |
| `(3, 5)` | `(3, 1)` | ✅ | one value per **row**, reused along it |

Try it on a small grid first.

In [ ]:
# DEMO - broadcasting on a grid you can read
toy = np.array([[ 1, -1, 3, 1,  0],
                [-1,  2, 2, 0,  1],
                [ 0, -1, 4, 2, -1]])
per_column = np.array([10, 20, 30, 40, 50])     # shape (5,)
per_row = np.array([100, 200, 300])             # shape (3,)

print(toy + per_column)                         # works: one value per column
print(toy + per_row[:, np.newaxis])             # works: (3,) became (3, 1), one value per row
try:
    toy + per_row                               # (3, 5) + (3,): the shapes don't line up
except ValueError as error:
    print("ValueError:", error)

### 🐞 Broken — the baseline that won't subtract

Run it and read the last line of the error: it tells you both shapes.

> 💡 One baseline per **row** — so it needs to be a column, shape `(254, 1)`. The table above shows how.

In [ ]:
# 🐞 BROKEN — subtract each go trial's own baseline
baseline = go_trials[:, :10].mean(axis=1)      # one per trial: shape (254,)
go_corrected = go_trials - baseline
print(go_corrected.shape)


# --- self-check (do not edit) ---
assert go_corrected.shape == (254, 40), f"❌ go_corrected should be (254, 40), got {go_corrected.shape}"
assert np.allclose(go_corrected[:, :10].mean(axis=1), 0), "❌ Each trial's baseline frames should now average 0 — subtract baseline[:, np.newaxis]"
print("✅ Fixed — (254, 40) minus (254, 1): each trial's baseline is reused along its own row")

`keepdims=True` is another way to the same place: `go_trials[:, :10].mean(axis=1, keepdims=True)` keeps the collapsed axis as size 1, so the answer is already `(254, 1)`.

## Stretch 4 — One score per trial, with `axis=1`

Notebook 2 averaged **down** the columns to get one curve. For a statistical test we need the opposite: **one number per trial** — how much that single trial responded.

Score each trial as the mean of its baseline-corrected frames **after** the start — columns 10 onwards, averaged **along the row**: `axis=1`.

### 🔬 Write it

**Your task:**

1. Make `nogo_corrected` — `nogo_trials` minus each trial's own baseline. (Use the 🐞 fix, or `keepdims=True`.)
2. Make `go_scores` and `nogo_scores` — one score per trial: the mean of the corrected frames from **column 10 onwards**.

> 💡 `go_corrected[:, 10:]` is every trial, from the start onwards. Then average along each row.

In [ ]:
# TODO — nogo_corrected, then go_scores and nogo_scores


# --- self-check (do not edit) ---
assert "nogo_corrected" in globals(), "❌ No variable called nogo_corrected yet"
assert nogo_corrected.shape == (263, 40) and np.allclose(nogo_corrected[:, :10].mean(axis=1), 0), "❌ nogo_corrected should be nogo_trials minus each trial's own baseline — same as the 🐞 fix"
assert "go_scores" in globals() and "nogo_scores" in globals(), "❌ Make go_scores and nogo_scores"
assert go_scores.shape != (30,), "❌ go_scores has one value per MOMENT — that's axis=0. You want one per trial: axis=1"
assert go_scores.shape == (254,), f"❌ go_scores should have one value per go trial, shape (254,), got {go_scores.shape}"
assert nogo_scores.shape == (263,), f"❌ nogo_scores should have shape (263,), got {nogo_scores.shape}"
assert round(go_scores.mean(), 3) == 0.102, f"❌ go_scores should average 0.102, got {round(go_scores.mean(), 3)} — the corrected frames from column 10 on"
assert round(nogo_scores.mean(), 3) == 0.007, f"❌ nogo_scores should average 0.007, got {round(nogo_scores.mean(), 3)}"
print("✅ Correct — on average, a go trial scores", round(go_scores.mean(), 3), "and a nogo trial", round(nogo_scores.mean(), 3))

## Stretch 5 🏆 — A shuffle test: is the difference real?

Go trials score 0.102 on average, nogo trials 0.007. The difference is about 0.096. Could **chance** give a difference that big?

Here is the argument, and it needs no statistics textbook:

- Suppose the neuron **doesn't care** which word it heard. Then the labels "go" and "nogo" mean nothing — any trial could have carried either one.
- So **shuffle** the labels: pour all 517 scores into one pile, mix them, and deal them back out as 254 "go" and 263 "nogo".
- Measure the difference. Do it **1,000 times**. That is what the difference looks like when only chance is at work.
- If the real difference is bigger than almost all of the shuffled ones, chance is a poor explanation. The fraction of shuffles that reach it is the **p-value**.

numpy's random generator does the mixing. Like `random` in the lecture — but for arrays, and with a **seed** so everyone gets the same answer:

```python
rng = np.random.default_rng(0)     # the seed: 0
rng.permutation(pooled)            # a shuffled copy
```

In [ ]:
# DEMO - the observed difference, and ONE shuffle
observed = go_scores.mean() - nogo_scores.mean()
print("observed difference:", round(observed, 3))

pooled = np.concatenate([go_scores, nogo_scores])    # all 517 scores in one array
n_go = len(go_scores)

rng = np.random.default_rng(0)
shuffled = rng.permutation(pooled)
fake_go, fake_nogo = shuffled[:n_go], shuffled[n_go:]
print("one shuffled difference:", round(fake_go.mean() - fake_nogo.mean(), 3))

### 🔬 Write it — 1,000 shuffles

**Your task:**

1. Shuffle **1,000** times, collecting each shuffled difference. Turn them into an array called `fake_diffs`.
2. Store the **fraction** of shuffled differences that are **at least as big** as `observed` in `p_value`.

> 💡 A loop is fine here — each shuffle needs fresh random numbers. Start from `fake_diffs = []`, append inside the loop, then `np.array()`.
> For 2: a mask's `.mean()` is the fraction of `True`s.

In [ ]:
# TODO — fake_diffs (1000 shuffled differences) and p_value


# --- self-check (do not edit) ---
assert "fake_diffs" in globals() and "p_value" in globals(), "❌ Make fake_diffs and p_value"
assert isinstance(fake_diffs, np.ndarray), "❌ fake_diffs should be an array — np.array(fake_diffs) after the loop"
assert fake_diffs.shape == (1000,), f"❌ fake_diffs should hold 1000 differences, got shape {fake_diffs.shape}"
assert fake_diffs.std() > 0, "❌ Every shuffled difference is the same — shuffle inside the loop"
assert abs(fake_diffs.mean()) < 0.01, "❌ Shuffled differences should centre on 0 — are you dealing out n_go 'go' scores and the rest 'nogo'?"
assert 0 <= p_value <= 1, f"❌ p_value is a fraction between 0 and 1, got {p_value}"
assert p_value == (fake_diffs >= observed).mean(), "❌ p_value should be the fraction of fake_diffs that are >= observed"
assert p_value < 0.01, f"❌ For neuron 1, p_value should be below 0.01, got {p_value}"
print(f"✅ Correct — not one shuffle in 1000 came close (the biggest was {fake_diffs.max():.3f}, the real one {observed:.3f}). Chance doesn't explain neuron 1.")

### 🏆 Finale — the whole test, in one function

**Your task:** define `shuffle_test(trace, go_marker, nogo_marker)` that runs Stretch tasks 1–5 on any neuron and **returns** `(observed, p_value)`. Then run it on **neuron 2**.

Inside, make your own generator with `np.random.default_rng(0)`, so the answer is the same every run.

> 💡 It's every cell above, joined up — with `trace`, `go_marker` and `nogo_marker` in place of `dff`, `go` and `nogo`. A small helper function `scores(trace, marker)` that returns one score per trial keeps it short: you need it twice.

In [ ]:
# TODO — define shuffle_test(trace, go_marker, nogo_marker), then run it on both neurons


# --- self-check (do not edit) ---
assert "shuffle_test" in globals(), "❌ No function called shuffle_test yet — check the name after def, and run the cell"
result_1 = shuffle_test(dff, go, nogo)
assert result_1 is not None, "❌ shuffle_test gives back None — did you forget to return?"
assert len(result_1) == 2, "❌ shuffle_test should return two things: (observed, p_value)"
assert round(result_1[0], 3) == round(observed, 3), f"❌ For neuron 1 the observed difference should be {observed:.3f}, got {result_1[0]:.3f}"
assert result_1[1] < 0.01, f"❌ For neuron 1, p should be below 0.01, got {result_1[1]}"
result_2 = shuffle_test(dff_2, go_2, nogo_2)
assert round(result_2[0], 3) != round(observed, 3), "❌ Neuron 2 gives neuron 1's answer — use trace, go_marker and nogo_marker inside the function"
assert round(result_2[0], 3) == -0.002, f"❌ For neuron 2 the observed difference should be -0.002, got {result_2[0]:.3f}"
assert result_2[1] > 0.2, f"❌ For neuron 2, p should be well above 0.05, got {result_2[1]}"
print(f"✅ Correct — neuron 1: p = {result_1[1]:.3f}. Neuron 2: p = {result_2[1]:.3f}, a difference chance makes all the time.")

## The verdict

- **Neuron 1:** none of 1,000 shuffles came near the real difference, so p < 0.001. It tells go from nogo.
- **Neuron 2:** responds strongly to both words, and its go−nogo difference is something shuffling produces about half the time. No evidence that it tells them apart.

The same conclusion as Part 1 — but now it is a **test**, not an impression. And the analysis checks its own assumption, needs no loop to cut the trials, and corrects every trial for its own baseline.

> This is a **permutation test**, and it is used in real papers. It makes almost no assumptions about the data — its only claim is "if the labels didn't matter, shuffling them wouldn't matter either".

## 🔭 Explore further

- `p_value` is **one-sided**: it asks whether go is *bigger*. Make it two-sided with `np.abs(fake_diffs) >= abs(observed)`.
- Change the score window: columns `20:` instead of `10:`. Does neuron 2's verdict change? Choosing a window *after* looking at the data is a classic way to fool yourself.
- `scipy.stats.permutation_test` does all of this in one call: <https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.permutation_test.html>
- Broadcasting, explained with pictures: <https://numpy.org/doc/stable/user/basics.broadcasting.html>
- The Lab 5 project, `analyse_neurons.py`, runs on **five** neurons. Its `onsets_of` and `go_response` functions use exactly the tools from this notebook — you can now read every line.

## ✅ You've got it if you can…

- [ ] Plan an analysis from a question, without a skeleton
- [ ] Spot the trial too close to the start before it breaks anything
- [ ] Write one function that analyses any neuron
- [ ] *Stretch:* find onsets with `np.diff`, cut windows without a loop, fix a broadcasting error, and explain what a shuffle test's p-value means